In [2]:
# Cell 1: imports + helpers (fixed-point, parsing)

from dataclasses import dataclass
from typing import List, Callable, Optional, Tuple, Iterable
import re
import numpy as np

def sign_extend(value: int, bits: int) -> int:
    """Interpret 'value' as signed integer with 'bits' width."""
    mask = (1 << bits) - 1
    value &= mask
    sign_bit = 1 << (bits - 1)
    return (value ^ sign_bit) - sign_bit

def sat_signed(value: int, bits: int) -> int:
    """Saturate to signed range of 'bits'."""
    lo = -(1 << (bits - 1))
    hi = (1 << (bits - 1)) - 1
    return lo if value < lo else hi if value > hi else value

def parse_int_maybe_hex(s: str) -> int:
    s = s.strip()

    # убрать хвостовые разделители типа "38," "38;" "38,"
    s = re.sub(r"[,\s;]+$", "", s)
    if s == "":
        raise ValueError("Empty token after stripping separators")

    if s.lower().startswith("0x"):
        return int(s, 16)

    # если похоже на hex без 0x (как твои 01A3), трактуем как hex
    if re.fullmatch(r"[0-9a-fA-F]+", s):
        # ВАЖНО: это сделает и чисто цифровые строки (например "38") тоже hex,
        # поэтому ниже мы добавим контекст в парсере спайков: спайки читаем как DEC.
        return int(s, 16)

    return int(s, 10)

def load_spike_timeslots_txt(path: str) -> List[List[int]]:
    """
    Спайки считаем ДЕСЯТИЧНЫМИ id пресинаптических нейронов.
    Форматы:
      - один id на строку (как у тебя), допускается "38,"
      - несколько id в строке: "0,2,5" или "0 2 5"
      - таймслоты можно разделять пустыми строками
    """
    with open(path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    timeslots: List[List[int]] = []
    cur: List[int] = []

    for ln in lines:
        ln = ln.strip()
        if not ln or ln.startswith("#"):
            if cur:
                timeslots.append(cur)
                cur = []
            continue

        # убрать inline comment
        ln = re.split(r"\s+#", ln)[0].strip()

        # разбить по запятым/пробелам
        parts = [p for p in re.split(r"[,\s]+", ln) if p != ""]
        for p in parts:
            # спайки = DEC, поэтому int(p,10) + обрезать хвостовые запятые
            p = re.sub(r"[,\s;]+$", "", p)
            if p == "":
                continue
            cur.append(int(p, 10))

    if cur:
        timeslots.append(cur)

    return timeslots

def load_weights_dat_hex(path: str, weight_bits: int) -> List[int]:
    """
    Веса считаем HEX (как у тебя: 01A3, 0131, ...).
    На строке может быть 0xXXXX или просто XXXX.
    """
    weights: List[int] = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("#"):
                continue
            ln = re.split(r"\s+#", ln)[0].strip()

            # убрать хвостовые запятые/точки с запятой на всякий случай
            ln = re.sub(r"[,\s;]+$", "", ln)
            if ln == "":
                continue

            # строго hex
            if ln.lower().startswith("0x"):
                w_u = int(ln, 16)
            else:
                w_u = int(ln, 16)

            w_s = sign_extend(w_u, weight_bits)
            weights.append(w_s)

    return weights

In [3]:
# Cell 2: configuration + model

@dataclass
class FixedPointCfg:
    acc_bits: int        # membrane / accumulator width
    frac_bits: int       # fractional bits (Q format: Q(acc_bits-frac_bits-1).frac_bits)
    weight_bits: int     # how weights are stored in hex file (signed)
    pre_count: int
    post_count: int

@dataclass
class LifParams:
    threshold: int       # fixed-point int
    reset_v: int         # fixed-point int
    leak_shift: int = 1  # default leak = arithmetic shift right by leak_shift

# Leak operator signature: takes vector v (np.int64) and returns leaked vector
LeakFn = Callable[[np.ndarray], np.ndarray]

def leak_shift_arith(v: np.ndarray, shift: int) -> np.ndarray:
    """
    Arithmetic right shift for signed ints in numpy.
    """
    # numpy right-shift is arithmetic for signed integers
    return (v >> shift)

class LifGoldenModel:
    def __init__(
        self,
        cfg: FixedPointCfg,
        lif: LifParams,
        weights_pre_post: np.ndarray,
        leak_fn: Optional[LeakFn] = None,
    ):
        """
        weights_pre_post shape: (pre_count, post_count), signed ints (weight_bits -> Python int)
        membrane state is signed fixed-point with acc_bits
        """
        assert weights_pre_post.shape == (cfg.pre_count, cfg.post_count)
        self.cfg = cfg
        self.lif = lif
        self.W = weights_pre_post.astype(np.int64)
        self.V = np.zeros(cfg.post_count, dtype=np.int64)  # membrane vector
        self.leak_fn = leak_fn if leak_fn is not None else (lambda x: leak_shift_arith(x, lif.leak_shift))

    def _sat_vec(self, v: np.ndarray) -> np.ndarray:
        lo = -(1 << (self.cfg.acc_bits - 1))
        hi = (1 << (self.cfg.acc_bits - 1)) - 1
        return np.clip(v, lo, hi).astype(np.int64)

    def reset_state(self, v_init: int = 0):
        self.V[:] = np.int64(sat_signed(v_init, self.cfg.acc_bits))

    def step_timeslot(
        self,
        spikes_pre: List[int],
        trace: bool = True,
        trace_neurons: Optional[Iterable[int]] = None,
    ) -> Tuple[List[int], dict]:
        """
        One timeslot:
          1) accumulate weighted contributions for each input spike
          2) leak
          3) threshold -> spikes_out
          4) reset spiking neurons to reset_v

        Returns:
          spikes_out: list of post neuron indices that spiked
          info: dict with detailed trace data
        """
        postN = self.cfg.post_count
        V_before = self.V.copy()

        if trace_neurons is None:
            trace_neurons = range(postN)
        trace_neurons = list(trace_neurons)

        info = {
            "V_before": V_before.copy(),
            "accum_events": [],  # per input spike
            "V_after_accum": None,
            "V_after_leak": None,
            "spikes_out": None,
            "V_after_reset": None,
        }

        # 1) accumulate per spike
        for k, pre_id in enumerate(spikes_pre):
            if pre_id < 0 or pre_id >= self.cfg.pre_count:
                raise ValueError(f"pre_id out of range: {pre_id} (0..{self.cfg.pre_count-1})")

            delta = self.W[pre_id, :]  # vector over posts
            V_prev = self.V.copy()
            self.V = self._sat_vec(self.V + delta)

            if trace:
                info["accum_events"].append({
                    "k": k,
                    "pre_id": int(pre_id),
                    "delta_sample": {int(i): int(delta[i]) for i in trace_neurons},
                    "V_prev_sample": {int(i): int(V_prev[i]) for i in trace_neurons},
                    "V_new_sample": {int(i): int(self.V[i]) for i in trace_neurons},
                })

        info["V_after_accum"] = self.V.copy()

        # 2) leak
        V_prev = self.V.copy()
        self.V = self._sat_vec(self.leak_fn(self.V))
        info["V_after_leak"] = self.V.copy()

        # 3) threshold
        thr = np.int64(self.lif.threshold)
        fired = (self.V >= thr)
        spikes_out = np.where(fired)[0].astype(int).tolist()
        info["spikes_out"] = spikes_out

        # 4) reset fired
        V_prev2 = self.V.copy()
        if spikes_out:
            self.V[fired] = np.int64(sat_signed(self.lif.reset_v, self.cfg.acc_bits))
        info["V_after_reset"] = self.V.copy()

        if trace:
            info["leak_step"] = {
                "V_prev_sample": {int(i): int(V_prev[i]) for i in trace_neurons},
                "V_new_sample": {int(i): int(self.V[i]) for i in trace_neurons},
                "leak_kind": "shift" if self.leak_fn.__name__ == "<lambda>" else getattr(self.leak_fn, "__name__", "custom"),
                "leak_shift": int(self.lif.leak_shift),
            }
            info["threshold_step"] = {
                "threshold": int(thr),
                "V_before_reset_sample": {int(i): int(V_prev2[i]) for i in trace_neurons},
                "fired_sample": {int(i): bool(fired[i]) for i in trace_neurons},
                "reset_v": int(self.lif.reset_v),
            }

        return spikes_out, info

In [4]:
# Cell 3: load your files and build weights matrix (pre-major order)
import os


# ---- CONFIG: set these to your design numbers ----
PRE_COUNT  = 28 * 28          # example
POST_COUNT = 10               # example

ACC_BITS    = 32              # membrane accumulator width in bits (match RTL)
FRAC_BITS   = 0               # set if you use Q format (e.g. 8, 12, 16)
WEIGHT_BITS = 16               # width of weights stored in hex file (signed)

# LIF params in fixed-point int
# If FRAC_BITS > 0, remember: real_value * (1<<FRAC_BITS) => fixed int
THRESHOLD = 100               # example fixed-point
RESET_V   = 0                 # example fixed-point
LEAK_SHIFT = 1                # divide by 2 each timeslot (arith shift)

SPIKES_TXT_PATH  = "/home/yan/nirsii/nncompiler/neuromorphic_snn/examples/1_mnist/data/input_queue/spikes_by_ticks/spiked_neurons_sample_1805_step0.txt"
WEIGHTS_DAT_PATH = "/home/yan/nirsii/nncompiler/neuromorphic_snn/examples/1_mnist/data/weights_fc1_1805.dat"

for p in (SPIKES_TXT_PATH, WEIGHTS_DAT_PATH):
    if not os.path.isfile(p):
        raise FileNotFoundError(f"File not found: {p}")

cfg = FixedPointCfg(
    acc_bits=ACC_BITS,
    frac_bits=FRAC_BITS,
    weight_bits=WEIGHT_BITS,
    pre_count=PRE_COUNT,
    post_count=POST_COUNT,
)

lif = LifParams(
    threshold=THRESHOLD,
    reset_v=RESET_V,
    leak_shift=LEAK_SHIFT,
)

timeslots = load_spike_timeslots_txt(SPIKES_TXT_PATH)
w_list = load_weights_dat_hex(WEIGHTS_DAT_PATH, weight_bits=WEIGHT_BITS)
print("first 8 weights signed:", w_list[:8])
print("first 8 weights raw hex:", [hex(int(x) & ((1<<WEIGHT_BITS)-1)) for x in w_list[:8]])

expected_len = PRE_COUNT * POST_COUNT
if len(w_list) != expected_len:
    raise ValueError(f"weights length mismatch: got {len(w_list)} expected {expected_len} (=pre*post)")

# pre-major reshape: [pre0(post0..), pre1(post0..), ...]
W = np.array(w_list, dtype=np.int64).reshape((PRE_COUNT, POST_COUNT))

model = LifGoldenModel(cfg=cfg, lif=lif, weights_pre_post=W)
model.reset_state(v_init=0)

len(timeslots), W.shape

first 8 weights signed: [419, 305, 483, 357, 147, 353, 385, 321]
first 8 weights raw hex: ['0x1a3', '0x131', '0x1e3', '0x165', '0x93', '0x161', '0x181', '0x141']


(1, (784, 10))

In [5]:
# Cell 4: run N timeslots with detailed trace

def pretty_trace(info: dict, trace_neurons: List[int]):
    print("V_before:", {i: int(info["V_before"][i]) for i in trace_neurons})

    for ev in info["accum_events"]:
        k = ev["k"]
        pre_id = ev["pre_id"]
        print(f"  after spike k={k} pre={pre_id}:")
        for i in trace_neurons:
            dv = ev["delta_sample"][i]
            vp = ev["V_prev_sample"][i]
            vn = ev["V_new_sample"][i]
            print(f"    post={i}: V {vp} + ({dv}) -> {vn}")

    print("V_after_accum:", {i: int(info["V_after_accum"][i]) for i in trace_neurons})
    print("V_after_leak :", {i: int(info["V_after_leak"][i]) for i in trace_neurons})
    print("spikes_out   :", info["spikes_out"])
    print("V_after_reset:", {i: int(info["V_after_reset"][i]) for i in trace_neurons})

TRACE_NEURONS = list(range(min(POST_COUNT, 8)))  # show first 8 posts

for t, spikes_pre in enumerate(timeslots[:10]):  # first 10 timeslots
    spikes_out, info = model.step_timeslot(spikes_pre, trace=True, trace_neurons=TRACE_NEURONS)
    print("="*80)
    print(f"TIMESLOT t={t}, spikes_pre={spikes_pre}")
    pretty_trace(info, TRACE_NEURONS)

TIMESLOT t=0, spikes_pre=[0, 2, 5, 6, 10, 12, 13, 14, 16, 19, 20, 21, 24, 25, 27, 28, 29, 30, 32, 37, 38, 39, 40, 41, 42, 47, 48, 49, 50, 51, 52, 53, 54, 56, 59, 63, 66, 67, 68, 69, 70, 73, 76, 77, 78, 79, 81, 82, 85, 87, 90, 91, 94, 96, 97, 99, 100, 102, 103, 104, 107, 108, 109, 110, 111, 112, 118, 119, 121, 123, 125, 127, 128, 129, 132, 134, 137, 139, 140, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 156, 158, 159, 161, 163, 165, 170, 171, 174, 175, 176, 177, 179, 182, 183, 184, 185, 187, 188, 189, 193, 195, 197, 199, 201, 202, 203, 204, 205, 206, 207, 209, 210, 211, 212, 213, 214, 215, 216, 222, 223, 225, 229, 230, 231, 232, 233, 235, 236, 238, 239, 240, 241, 242, 243, 244, 245, 246, 249, 250, 252, 253, 255, 256, 257, 258, 259, 260, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 277, 278, 279, 280, 281, 285, 286, 287, 288, 289, 294, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 313, 314, 315, 316, 317, 318, 320, 321, 322, 324, 325, 327, 328, 329, 330,

In [6]:
# Cell 5: make it convenient to compare with RTL dumps (optional)

def run_all_timeslots(model: LifGoldenModel, timeslots: List[List[int]]) -> List[List[int]]:
    out = []
    for spikes_pre in timeslots:
        spikes_out, _ = model.step_timeslot(spikes_pre, trace=False)
        out.append(spikes_out)
    return out

# Example: get all outputs
model.reset_state(0)
all_spikes_out = run_all_timeslots(model, timeslots)

# Save to file (one timeslot per line)
with open("golden_spikes_out.txt", "w", encoding="utf-8") as f:
    for t, s in enumerate(all_spikes_out):
        f.write(" ".join(map(str, s)) + "\n")

len(all_spikes_out), all_spikes_out[:5]

(1, [[0, 1, 9]])

In [7]:
model.reset_state(v_init=0)

for t, spikes_pre in enumerate(timeslots):
    spikes_out, info = model.step_timeslot(
        spikes_pre,
        trace=True,
        trace_neurons=TRACE_NEURONS
    )

In [8]:
SPIKES_TXT_PATH  = "/home/yan/nirsii/nncompiler/neuromorphic_snn/examples/1_mnist/data/input_queue/spikes_by_ticks/spiked_neurons_sample_1805_step0.txt"
WEIGHTS_DAT_PATH = "/home/yan/nirsii/nncompiler/neuromorphic_snn/examples/1_mnist/data/weights_fc1_1805.dat"

timeslots = load_spike_timeslots_txt(SPIKES_TXT_PATH)
print("timeslots:", len(timeslots))
print("first timeslot spikes count:", len(timeslots[0]) if timeslots else 0)
print("first 50 spikes:", timeslots[0][:50] if timeslots else [])

w_list = load_weights_dat_hex(WEIGHTS_DAT_PATH, weight_bits=WEIGHT_BITS)
print("weights:", len(w_list))
print("first 16 weights (signed):", w_list[:16])
print("first 16 weights (hex u):", [hex(int(x) & ((1<<WEIGHT_BITS)-1)) for x in w_list[:16]])

timeslots: 1
first timeslot spikes count: 505
first 50 spikes: [0, 2, 5, 6, 10, 12, 13, 14, 16, 19, 20, 21, 24, 25, 27, 28, 29, 30, 32, 37, 38, 39, 40, 41, 42, 47, 48, 49, 50, 51, 52, 53, 54, 56, 59, 63, 66, 67, 68, 69, 70, 73, 76, 77, 78, 79, 81, 82, 85, 87]
weights: 7840
first 16 weights (signed): [419, 305, 483, 357, 147, 353, 385, 321, 466, 344, 446, 470, 278, 429, 343, 419]
first 16 weights (hex u): ['0x1a3', '0x131', '0x1e3', '0x165', '0x93', '0x161', '0x181', '0x141', '0x1d2', '0x158', '0x1be', '0x1d6', '0x116', '0x1ad', '0x157', '0x1a3']
